In [ ]:
# ============================================================
# AICW-NASA-TechPort-Portfolio
# End-to-end Colab notebook script
# ============================================================

import os
import time
import math
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from scipy.stats import kendalltau, spearmanr

# ------------------------------------------------------------
# 0. USER SETTINGS
# ------------------------------------------------------------
# If techport_projects_batch1.csv exists, script uses it directly.
# Otherwise it tries to build it from api_urls.csv via NASA TechPort API.

USE_EXISTING_BATCH = True
MAX_API_PROJECTS = 300   # only used if techport_projects_batch1.csv is missing
API_SLEEP = 0.2

RAW_BATCH_FILE = "techport_projects_batch1.csv"
API_URLS_FILE = "api_urls.csv"

# ------------------------------------------------------------
# 1. OPTIONAL: FETCH NASA TECHPORT DATA IF BATCH FILE MISSING
# ------------------------------------------------------------
if (not os.path.exists(RAW_BATCH_FILE)) and os.path.exists(API_URLS_FILE):
    print(f"{RAW_BATCH_FILE} bulunamadı. {API_URLS_FILE} üzerinden veri çekiliyor...")

    urls = pd.read_csv(API_URLS_FILE, header=None)[0].dropna()
    urls = urls[urls.str.startswith("https://techport.nasa.gov/api/projects/")].drop_duplicates()

    records = []

    for url in urls.iloc[:MAX_API_PROJECTS]:
        try:
            r = requests.get(url, timeout=30)
            if r.status_code != 200:
                continue

            data = r.json()
            obj = data.get("project", {})

            records.append({
                "projectId": obj.get("projectId"),
                "title": obj.get("title"),
                "startDate": obj.get("startDate"),
                "endDate": obj.get("endDate"),
                "status": obj.get("status"),
                "releaseStatus": obj.get("releaseStatus"),
                "trlBegin": obj.get("trlBegin"),
                "trlCurrent": obj.get("trlCurrent"),
                "trlEnd": obj.get("trlEnd"),
                "programTitle": (obj.get("program") or {}).get("title"),
                "missionDirectorate": ((obj.get("program") or {}).get("responsibleMd") or {}).get("acronym"),
                "primaryTxCode": (obj.get("primaryTx") or {}).get("code"),
                "primaryTxTitle": (obj.get("primaryTx") or {}).get("title"),
                "primaryTxLevel": (obj.get("primaryTx") or {}).get("level"),
                "otherOrgCount": len(obj.get("otherOrganizations") or []),
                "leadOrg": (obj.get("leadOrganization") or {}).get("acronym"),
            })

            time.sleep(API_SLEEP)

        except Exception as e:
            print("API error:", e)

    batch_df = pd.DataFrame(records)
    batch_df.to_csv(RAW_BATCH_FILE, index=False)
    print(f"{RAW_BATCH_FILE} oluşturuldu. Satır sayısı:", len(batch_df))

elif os.path.exists(RAW_BATCH_FILE):
    print(f"{RAW_BATCH_FILE} bulundu. Mevcut dosya kullanılacak.")
else:
    raise FileNotFoundError(
        "Ne techport_projects_batch1.csv ne de api_urls.csv bulundu. "
        "Lütfen bu dosyalardan en az birini Colab'e yükle."
    )

# ------------------------------------------------------------
# 2. LOAD RAW DATA
# ------------------------------------------------------------
raw = pd.read_csv(RAW_BATCH_FILE)
print("Raw data shape:", raw.shape)

# ------------------------------------------------------------
# 3. BUILD DECISION MATRIX
# ------------------------------------------------------------
df = raw.copy()

# date parsing
df["startDate"] = pd.to_datetime(df["startDate"], errors="coerce")
df["endDate"] = pd.to_datetime(df["endDate"], errors="coerce")

# duration in months
df["duration_months"] = (df["endDate"] - df["startDate"]).dt.days / 30.0

# trl gap
df["trl_gap"] = df["trlEnd"] - df["trlCurrent"]

# complete-case filtering variables
required_cols = [
    "projectId",
    "trlCurrent",
    "duration_months",
    "trl_gap",
    "primaryTxLevel",
    "otherOrgCount",
    "trlBegin",
    "primaryTxCode"
]

filtered = df.dropna(subset=required_cols).copy()

# taxonomy rarity
freq = filtered["primaryTxCode"].value_counts()
filtered["taxonomy_rarity"] = filtered["primaryTxCode"].map(lambda x: 1 / freq[x])

# AICW target: observed technology maturation
filtered["TRL_progress"] = filtered["trlCurrent"] - filtered["trlBegin"]

# decision matrix
decision_matrix = filtered[[
    "projectId",
    "trlCurrent",
    "duration_months",
    "trl_gap",
    "primaryTxLevel",
    "taxonomy_rarity",
    "otherOrgCount"
]].copy()

decision_matrix.to_csv("decision_matrix.csv", index=False)
print("decision_matrix.csv oluşturuldu. Shape:", decision_matrix.shape)

# ------------------------------------------------------------
# 4. DATASET SUMMARY
# ------------------------------------------------------------
dataset_summary = pd.DataFrame({
    "Metric": [
        "Initial project records",
        "Filtered complete-case sample",
        "Retention ratio",
        "Number of decision criteria",
        "Data source"
    ],
    "Value": [
        len(raw),
        len(decision_matrix),
        round(len(decision_matrix) / len(raw), 4) if len(raw) > 0 else np.nan,
        6,
        "NASA TechPort API"
    ]
})
dataset_summary.to_csv("dataset_summary.csv", index=False)
print("dataset_summary.csv oluşturuldu.")

# ------------------------------------------------------------
# 5. PREPARE FEATURES
# ------------------------------------------------------------
criteria = [
    "trlCurrent",
    "duration_months",
    "trl_gap",
    "primaryTxLevel",
    "taxonomy_rarity",
    "otherOrgCount"
]

benefit_criteria = [
    "trlCurrent",
    "trl_gap",
    "primaryTxLevel",
    "taxonomy_rarity",
    "otherOrgCount"
]

cost_criteria = ["duration_months"]

X = decision_matrix[criteria].copy()
y = filtered["TRL_progress"].copy()

def normalize_matrix(matrix):
    norm = matrix.copy().astype(float)
    for c in norm.columns:
        maxv = norm[c].max()
        minv = norm[c].min()

        if pd.isna(maxv) or pd.isna(minv):
            norm[c] = np.nan
        elif maxv == minv:
            norm[c] = 1.0
        elif c in benefit_criteria:
            norm[c] = (norm[c] - minv) / (maxv - minv)
        elif c in cost_criteria:
            norm[c] = (maxv - norm[c]) / (maxv - minv)
        else:
            norm[c] = (norm[c] - minv) / (maxv - minv)

    return norm

X_norm = normalize_matrix(X)
X_norm = X_norm.fillna(0) + 1e-9

# ------------------------------------------------------------
# 6. WEIGHTING METHODS
# ------------------------------------------------------------

# 6.1 AICW
rf = RandomForestRegressor(n_estimators=500, random_state=42)
rf.fit(X_norm, y)
aicw_weights = rf.feature_importances_
aicw_weights = aicw_weights / aicw_weights.sum()

# 6.2 Standard Deviation weighting
sd = X_norm.std()
sd_weights = (sd / sd.sum()).values

# 6.3 MEREC (stable implementation)
merec_matrix = X_norm.copy() + 1e-9
S_all = np.abs(np.log(merec_matrix)).sum(axis=1)

merec_effects = []
for c in merec_matrix.columns:
    reduced = merec_matrix.drop(columns=[c])
    S_reduced = np.abs(np.log(reduced)).sum(axis=1)
    E_j = np.abs(S_all - S_reduced).sum()
    merec_effects.append(E_j)

merec_weights = np.array(merec_effects, dtype=float)
merec_weights = merec_weights / merec_weights.sum()

# 6.4 FUCOM (benchmark parameterization, not expert elicitation)
# priority order:
# trlCurrent > trl_gap > duration_months > primaryTxLevel > taxonomy_rarity > otherOrgCount
fucom_order = ["trlCurrent", "trl_gap", "duration_months", "primaryTxLevel", "taxonomy_rarity", "otherOrgCount"]
phi = [1.0, 1.2, 1.25, 1.5, 1.4]

fucom_temp = [1.0]
for r in phi:
    fucom_temp.append(fucom_temp[-1] / r)

fucom_temp = np.array(fucom_temp, dtype=float)
fucom_temp = fucom_temp / fucom_temp.sum()

fucom_dict = {c: w for c, w in zip(fucom_order, fucom_temp)}
fucom_weights = np.array([fucom_dict[c] for c in criteria])

# 6.5 LBWA (benchmark parameterization)
lbwa_levels = {
    "trlCurrent": 1,
    "trl_gap": 2,
    "duration_months": 3,
    "primaryTxLevel": 4,
    "taxonomy_rarity": 4,
    "otherOrgCount": 5
}

lbwa_raw = np.array([1 / lbwa_levels[c] for c in criteria], dtype=float)
lbwa_weights = lbwa_raw / lbwa_raw.sum()

# save weights
weights_table = pd.DataFrame({
    "criterion": criteria,
    "AICW": aicw_weights,
    "FUCOM": fucom_weights,
    "LBWA": lbwa_weights,
    "MEREC": merec_weights,
    "SD": sd_weights
})
weights_table.to_csv("weights_table.csv", index=False)
print("weights_table.csv oluşturuldu.")

# ------------------------------------------------------------
# 7. RANKING METHODS
# ------------------------------------------------------------
def waspas(matrix, w, lam=0.5):
    M = np.asarray(matrix, dtype=float)
    w = np.asarray(w, dtype=float)

    wsm = (M * w).sum(axis=1)
    wpm = np.prod(np.power(M, w), axis=1)
    return lam * wsm + (1 - lam) * wpm

def marcos(matrix, w):
    M = np.asarray(matrix, dtype=float)
    w = np.asarray(w, dtype=float)

    ideal = M.max(axis=0)
    anti = M.min(axis=0)

    S = (M * w).sum(axis=1)
    S_ideal = np.sum(ideal * w)
    S_anti = np.sum(anti * w)

    K_plus = S / (S_ideal + 1e-9)
    K_minus = S / (S_anti + 1e-9)

    utility = (K_plus + K_minus) / 2.0
    return utility

ranking_methods = {
    "AICW": weights_table["AICW"].values.astype(float),
    "FUCOM": weights_table["FUCOM"].values.astype(float),
    "LBWA": weights_table["LBWA"].values.astype(float),
    "MEREC": weights_table["MEREC"].values.astype(float),
    "SD": weights_table["SD"].values.astype(float)
}

# WASPAS ranking
ranking_waspas = pd.DataFrame(index=decision_matrix["projectId"].values)

for m, w in ranking_methods.items():
    score = waspas(X_norm.values, w)
    ranking_waspas[m + "_score"] = score
    ranking_waspas[m + "_rank"] = pd.Series(score, index=ranking_waspas.index).rank(ascending=False, method="min")

ranking_waspas.to_csv("ranking_waspas_fixed.csv")
print("ranking_waspas_fixed.csv oluşturuldu.")

# MARCOS ranking
ranking_marcos = pd.DataFrame(index=decision_matrix["projectId"].values)

for m, w in ranking_methods.items():
    score = marcos(X_norm.values, w)
    ranking_marcos[m + "_score"] = score
    ranking_marcos[m + "_rank"] = pd.Series(score, index=ranking_marcos.index).rank(ascending=False, method="min")

ranking_marcos.to_csv("ranking_marcos_fixed.csv")
print("ranking_marcos_fixed.csv oluşturuldu.")

# ------------------------------------------------------------
# 8. CONCORDANCE
# ------------------------------------------------------------
method_names = ["AICW", "FUCOM", "LBWA", "MEREC", "SD"]

def concordance_table(rank_df):
    rows = []
    for i in range(len(method_names)):
        for j in range(i + 1, len(method_names)):
            a = rank_df[method_names[i] + "_rank"]
            b = rank_df[method_names[j] + "_rank"]

            tau = kendalltau(a, b)[0]
            rho = spearmanr(a, b)[0]

            top_a = set(rank_df.sort_values(method_names[i] + "_rank").head(10).index)
            top_b = set(rank_df.sort_values(method_names[j] + "_rank").head(10).index)
            overlap = len(top_a & top_b)

            rows.append([method_names[i], method_names[j], tau, rho, overlap])
    return pd.DataFrame(rows, columns=["method1", "method2", "kendall_tau", "spearman_rho", "top10_overlap"])

cw = concordance_table(ranking_waspas)
cw["ranking_model"] = "WASPAS"

cm = concordance_table(ranking_marcos)
cm["ranking_model"] = "MARCOS"

concordance = pd.concat([cw, cm], ignore_index=True)
concordance.to_csv("concordance_table_fixed.csv", index=False)
print("concordance_table_fixed.csv oluşturuldu.")

# ------------------------------------------------------------
# 9. TOP-10 TABLES
# ------------------------------------------------------------
def make_top10(rank_df):
    out = pd.DataFrame({"Rank": range(1, 11)})
    for m in method_names:
        out[m] = rank_df.sort_values(m + "_rank").head(10).index.tolist()
    return out

top10_waspas = make_top10(ranking_waspas)
top10_marcos = make_top10(ranking_marcos)

top10_waspas.to_csv("top10_waspas_fixed.csv", index=False)
top10_marcos.to_csv("top10_marcos_fixed.csv", index=False)

print("top10_waspas_fixed.csv oluşturuldu.")
print("top10_marcos_fixed.csv oluşturuldu.")

# ------------------------------------------------------------
# 10. SENSITIVITY ANALYSIS (AICW)
# ------------------------------------------------------------
base_w = weights_table["AICW"].values.astype(float)
base_scores = waspas(X_norm.values, base_w)
base_rank = pd.Series(base_scores).rank(ascending=False, method="min")

sens_rows = []

for i, c in enumerate(criteria):
    w = base_w.copy()
    w[i] = w[i] * 1.10
    w = w / w.sum()

    pert_scores = waspas(X_norm.values, w)
    pert_rank = pd.Series(pert_scores).rank(ascending=False, method="min")

    mean_shift = np.mean(np.abs(base_rank - pert_rank))
    max_shift = np.max(np.abs(base_rank - pert_rank))

    sens_rows.append([c, mean_shift, max_shift])

sensitivity_table = pd.DataFrame(sens_rows, columns=["criterion", "mean_rank_shift", "max_rank_shift"])
sensitivity_table.to_csv("sensitivity_table.csv", index=False)
print("sensitivity_table.csv oluşturuldu.")

# ------------------------------------------------------------
# 11. FIGURES
# ------------------------------------------------------------

# Figure 1: workflow
fig, ax = plt.subplots(figsize=(12, 2.8))
ax.axis("off")

steps = [
    "NASA TechPort API",
    "Filtering &\ncomplete-case sample",
    "Criterion\nconstruction",
    "Weighting methods\n(AICW, FUCOM,\nLBWA, MEREC, SD)",
    "Ranking models\n(WASPAS, MARCOS)",
    "Concordance &\nsensitivity"
]

x_positions = np.linspace(0.07, 0.93, len(steps))

for i, (x, s) in enumerate(zip(x_positions, steps)):
    ax.text(
        x, 0.5, s,
        ha="center", va="center",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.45", edgecolor="black", facecolor="white")
    )
    if i < len(steps) - 1:
        ax.annotate(
            "", xy=(x_positions[i+1]-0.06, 0.5), xytext=(x+0.06, 0.5),
            arrowprops=dict(arrowstyle="->", lw=1.5)
        )

plt.tight_layout()
plt.savefig("Figure1_workflow.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 2: weights
fig, ax = plt.subplots(figsize=(11, 6))
plot_criteria = weights_table["criterion"].tolist()
plot_methods = [c for c in weights_table.columns if c != "criterion"]

x = np.arange(len(plot_criteria))
bar_width = 0.16

for i, m in enumerate(plot_methods):
    ax.bar(x + i * bar_width, weights_table[m].values, width=bar_width, label=m)

ax.set_xticks(x + bar_width * (len(plot_methods)-1) / 2)
ax.set_xticklabels(plot_criteria, rotation=30, ha="right")
ax.set_ylabel("Weight")
ax.set_title("Criterion weights across methods")
ax.legend()

plt.tight_layout()
plt.savefig("Figure2_weights.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 3: concordance heatmaps
method_list = sorted(set(concordance["method1"]).union(set(concordance["method2"])))

def build_matrix(df_sub, value_col):
    mat = pd.DataFrame(np.eye(len(method_list)), index=method_list, columns=method_list)
    for _, row in df_sub.iterrows():
        m1, m2 = row["method1"], row["method2"]
        mat.loc[m1, m2] = row[value_col]
        mat.loc[m2, m1] = row[value_col]
    return mat

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model in zip(axes, ["WASPAS", "MARCOS"]):
    sub = concordance[concordance["ranking_model"] == model]
    mat = build_matrix(sub, "kendall_tau")
    im = ax.imshow(mat.values)
    ax.set_xticks(range(len(method_list)))
    ax.set_xticklabels(method_list, rotation=45, ha="right")
    ax.set_yticks(range(len(method_list)))
    ax.set_yticklabels(method_list)
    ax.set_title(f"{model} concordance (Kendall τ)")
    for i in range(len(method_list)):
        for j in range(len(method_list)):
            ax.text(j, i, f"{mat.values[i, j]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
plt.tight_layout()
plt.savefig("Figure3_concordance.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 4: top-10 overlap
method_list = [c for c in top10_waspas.columns if c != "Rank"]

def overlap_counts(df_top):
    rows = []
    for i in range(len(method_list)):
        for j in range(i + 1, len(method_list)):
            set_i = set(df_top[method_list[i]].tolist())
            set_j = set(df_top[method_list[j]].tolist())
            rows.append([f"{method_list[i]} vs {method_list[j]}", len(set_i & set_j)])
    return pd.DataFrame(rows, columns=["comparison", "overlap"])

ov_w = overlap_counts(top10_waspas)
ov_m = overlap_counts(top10_marcos)

fig, axes = plt.subplots(2, 1, figsize=(11, 8))

axes[0].barh(ov_w["comparison"], ov_w["overlap"])
axes[0].set_title("Top-10 project overlap under WASPAS")
axes[0].set_xlabel("Overlap count")

axes[1].barh(ov_m["comparison"], ov_m["overlap"])
axes[1].set_title("Top-10 project overlap under MARCOS")
axes[1].set_xlabel("Overlap count")

plt.tight_layout()
plt.savefig("Figure4_top10_overlap.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 5: sensitivity
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(range(len(sensitivity_table)), sensitivity_table["mean_rank_shift"].values)
ax.set_ylabel("Mean rank shift")
ax.set_title("Sensitivity of AICW rankings to +10% one-at-a-time weight perturbation")
ax.set_xticks(range(len(sensitivity_table)))
ax.set_xticklabels(sensitivity_table["criterion"], rotation=30, ha="right")

plt.tight_layout()
plt.savefig("Figure5_sensitivity.png", dpi=300, bbox_inches="tight")
plt.close()

print("Figure1_workflow.png oluşturuldu.")
print("Figure2_weights.png oluşturuldu.")
print("Figure3_concordance.png oluşturuldu.")
print("Figure4_top10_overlap.png oluşturuldu.")
print("Figure5_sensitivity.png oluşturuldu.")

# ------------------------------------------------------------
# 12. FINAL CHECK
# ------------------------------------------------------------
print("\nTüm çıktı dosyaları hazır:")
outputs = [
    "decision_matrix.csv",
    "dataset_summary.csv",
    "weights_table.csv",
    "ranking_waspas_fixed.csv",
    "ranking_marcos_fixed.csv",
    "concordance_table_fixed.csv",
    "top10_waspas_fixed.csv",
    "top10_marcos_fixed.csv",
    "sensitivity_table.csv",
    "Figure1_workflow.png",
    "Figure2_weights.png",
    "Figure3_concordance.png",
    "Figure4_top10_overlap.png",
    "Figure5_sensitivity.png"
]

for f in outputs:
    print(f, "->", "OK" if os.path.exists(f) else "MISSING")

techport_projects_batch1.csv bulundu. Mevcut dosya kullanılacak.
Raw data shape: (299, 16)
decision_matrix.csv oluşturuldu. Shape: (214, 7)
dataset_summary.csv oluşturuldu.
weights_table.csv oluşturuldu.
ranking_waspas_fixed.csv oluşturuldu.
ranking_marcos_fixed.csv oluşturuldu.
concordance_table_fixed.csv oluşturuldu.
top10_waspas_fixed.csv oluşturuldu.
top10_marcos_fixed.csv oluşturuldu.
sensitivity_table.csv oluşturuldu.


/tmp/ipykernel_719/4285720057.py:499: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Figure1_workflow.png oluşturuldu.
Figure2_weights.png oluşturuldu.
Figure3_concordance.png oluşturuldu.
Figure4_top10_overlap.png oluşturuldu.
Figure5_sensitivity.png oluşturuldu.

Tüm çıktı dosyaları hazır:
decision_matrix.csv -> OK
dataset_summary.csv -> OK
weights_table.csv -> OK
ranking_waspas_fixed.csv -> OK
ranking_marcos_fixed.csv -> OK
concordance_table_fixed.csv -> OK
top10_waspas_fixed.csv -> OK
top10_marcos_fixed.csv -> OK
sensitivity_table.csv -> OK
Figure1_workflow.png -> OK
Figure2_weights.png -> OK
Figure3_concordance.png -> OK
Figure4_top10_overlap.png -> OK
Figure5_sensitivity.png -> OK
